In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import ml_tools as mlt
import sim_ranking as sr

In [ ]:
robin_data_dir = Path("/Users/claudy/dev/work/data/sim_ranking/other/robin_2024_emp_gmm_study_data/Br_10_rrup_fmin_mw_cmt_HNBN_newrrup_1p0_crustal_mw3p5_rrup300_0p12")
nzgmdb_ffp = Path("/Users/claudy/dev/work/data/gm_datasets/nz_gmdb/v4.2/custom/mod_ground_motion_im_table_rotd50_flat.csv")
emp_gm_params_ffp = Path("/Users/claudy/dev/work/data/sim_ranking/emp_gm_params/nzgmdb_v4p2/emp_gm_params_400MaxRjb.parquet")
gnn_results_dir = Path("/Users/claudy/dev/work/data/sim_ranking/results/gnn/0306_1947_cv_v4p2NZGMDB_v2p3_4e8s_fixedZ1p0Issue")

In [ ]:
tect_type_mapping = {"crustal": "Crustal", "subduction_interface": "SI", "subduction_slab": "SS", "unknown": "Unknown", "outer_rise": "Outer Rise"}

In [ ]:
gnn_val_results = pd.read_parquet(gnn_results_dir / "val_results.parquet")

In [ ]:
# Combine robin empirical GM parameters
r_events_df = pd.read_csv(robin_data_dir / "events.csv", index_col=0)
r_site_df = pd.read_csv(robin_data_dir / "stations.csv", index_col=0)
r_emp_gm_params = pd.read_csv(robin_data_dir / "im_sim.csv", dtype={"event_id": int,  "stat_id": int})

r_emp_gm_params["site"] = r_site_df.loc[r_emp_gm_params.stat_id, "sta"].values.astype(str)
r_emp_gm_params["event"] = r_events_df.loc[r_emp_gm_params.event_id]["evid"].values.astype(str)

r_emp_gm_params = r_emp_gm_params.drop(columns=["stat_id", "event_id"])
r_emp_gm_params = r_emp_gm_params.rename(columns={"site": "site_id", "event": "event_id"})

r_emp_gm_params.index = mlt.array_utils.numpy_str_join("_", r_emp_gm_params["event_id"].values.astype(str), r_emp_gm_params["site_id"].values.astype(str))

In [ ]:
# Combine Robin Observed data
r_obs_df = pd.read_csv(robin_data_dir / "im_obs.csv", dtype={"event_id": int,  "stat_id": int})
r_obs_df["site"] = r_site_df.loc[r_obs_df.stat_id, "sta"].values.astype(str)
r_obs_df["event"] = r_events_df.loc[r_obs_df.event_id]["evid"].values.astype(str)
r_obs_df = r_obs_df.drop(columns=["stat_id", "event_id"])
r_obs_df = r_obs_df.rename(columns={"site": "site_id", "event": "event_id"})
r_obs_df.index = mlt.array_utils.numpy_str_join("_", r_obs_df["event_id"].values.astype(str), r_obs_df["site_id"].values.astype(str))
r_obs_df = r_obs_df.drop(columns=["gm_id"])

assert r_obs_df.index.equals(r_emp_gm_params.index)

In [ ]:
# Compute overlapping IM keys
im_keys = np.intersect1d(sr.constants.PSA_KEYS, [cur_col for cur_col in r_emp_gm_params.columns if cur_col.startswith("pSA")])
periods = sorted([float(cur_key.rsplit("_", maxsplit=1)[1]) for cur_key in im_keys])
im_keys = mlt.array_utils.numpy_str_join("_", "pSA", np.asarray(periods).astype(str))
pred_im_keys = mlt.array_utils.numpy_str_join("_", im_keys, "pred")

pred_cols = ["event_id", "site_id"] + list(pred_im_keys)
obs_cols = ["event_id", "site_id"] + list(im_keys)

In [ ]:
# Robin data preparation
r_emp_gm_params = r_emp_gm_params.rename(columns=dict(zip(im_keys, pred_im_keys)))
r_emp_gm_params = r_emp_gm_params[pred_cols]
r_emp_gm_params[pred_im_keys] = np.log(r_emp_gm_params[pred_im_keys].values)
r_nan_mask = r_emp_gm_params[pred_im_keys].isna().values

r_obs_df = r_obs_df[obs_cols]
r_obs_df[im_keys] = np.log(r_obs_df[im_keys].values)

# Match the nan values
r_obs_im_values = r_obs_df[im_keys].values
r_obs_im_values[r_nan_mask] = np.nan
r_obs_df[im_keys] = r_obs_im_values

assert np.all(r_emp_gm_params[pred_im_keys].isna().values == r_obs_df[im_keys].isna().values)

In [ ]:
# Load Robin flatfile
r_nzmgbd_df = pd.read_csv(robin_data_dir / "ground_motion_im_table_rotd50_flat_variablerrup_cmt_HNBN_crustal_1p0_mw3p5_rrup300.csv", index_col=0)
r_nzmgbd_df.index = mlt.array_utils.numpy_str_join("_", r_nzmgbd_df["evid"].values.astype(str), r_nzmgbd_df["sta"].values.astype(str))

In [ ]:
# Load NZGMDB empirical GM params
emp_gm_params = pd.read_parquet(emp_gm_params_ffp)
emp_gm_params = emp_gm_params.rename(columns=dict(zip(sr.constants.GMM_PRED_PSA_KEYS, pred_im_keys)))
# emp_gm_params = emp_gm_params.drop(columns=[cur_col for cur_col in emp_gm_params.columns if cur_col not in im_keys])
emp_gm_params = emp_gm_params[pred_cols]

In [ ]:
# Determine shared record ids
shared_gm_records = np.intersect1d(emp_gm_params.index, r_emp_gm_params.index)

In [ ]:
# Load Observed NZGMDB data (4.2)
obs_data = sr.data.load_obs_nzgmdb(nzgmdb_ffp)

In [ ]:
# Load Observed NZGMDB data (4.0)
obs_data_v4p0 = sr.data.load_obs_nzgmdb(Path("/Users/claudy/dev/work/data/gm_datasets/nz_gmdb/v4.0/Tables/ground_motion_im_table_rotd50_flat.csv"))

In [ ]:
# Load Observed NZGMDB data (4.1)
obs_data_v4p1 = sr.data.load_obs_nzgmdb(Path("/Users/claudy/dev/work/data/gm_datasets/nz_gmdb/v4.1/Tables/ground_motion_im_table_rotd50_flat.csv"))

In [ ]:
# Load Robin Mixed Effects Regression Results
mera_df = pd.read_csv(robin_data_dir / "Residuals" / "PJSvarCompsBiased_sim.csv", index_col=0)

## Bias & Residual Standard Deviation

In [ ]:
# NZGMDB 4.2 Empirical GM Parameters (CS) + NZGMDB 4.2 Observed Residuals (Shared records)
cur_cs_n4_emp_gm_params = emp_gm_params.loc[shared_gm_records].copy(deep=True)
cur_cs_n4_emp_gm_params[im_keys] = np.log(obs_data.record_df.loc[cur_cs_n4_emp_gm_params.index.values, im_keys].values)

cur_cs_n4_res_df = sr.ml.gnn_gm.get_residuals(cur_cs_n4_emp_gm_params, im_keys, site_col="site_id")
cur_cs_n4_bias, cur_cs_n4_std = cur_cs_n4_res_df[im_keys].mean(axis=0), cur_cs_n4_res_df[im_keys].std(axis=0)

In [ ]:
# NZGMDB 4.2 Empirical GM Parameters (CS) + NZGMDB 1.0 Observed Residuals (Shared records)
cur_cs_n1_emp_gm_params = emp_gm_params.loc[shared_gm_records].copy(deep=True)
cur_cs_n1_emp_gm_params[im_keys] = r_obs_df.loc[shared_gm_records, im_keys].values

cur_cs_n1_res_df = sr.ml.gnn_gm.get_residuals(cur_cs_n1_emp_gm_params, im_keys, site_col="site_id")
cur_cs_n1_bias, cur_cs_n1_std = cur_cs_n1_res_df[im_keys].mean(axis=0), cur_cs_n1_res_df[im_keys].std(axis=0)

In [ ]:
# NZGMDB 1.0 Empirical GM Parameters (Robin) + NZGMDB 4.2 Observed Residuals (Shared records)
cur_r_n4_emp_gm_params = r_emp_gm_params.loc[shared_gm_records].copy(deep=True)
cur_r_n4_emp_gm_params[im_keys] = np.log(obs_data.record_df.loc[cur_r_n4_emp_gm_params.index.values, im_keys].values)

cur_r_n4_res_df = sr.ml.gnn_gm.get_residuals(cur_r_n4_emp_gm_params, im_keys, site_col="site_id")
cur_r_n4_bias, cur_r_n4_std = cur_r_n4_res_df[im_keys].mean(axis=0), cur_r_n4_res_df[im_keys].std(axis=0)

In [ ]:
# NZGMDB 1.0 Empirical GM Parameters (Robin) + NZGMDB 1.0 Observed Residuals (Shared records)
cur_r_n1_emp_gm_params = r_emp_gm_params.loc[shared_gm_records].copy(deep=True)
cur_r_n1_emp_gm_params[im_keys] = r_obs_df.loc[cur_r_n1_emp_gm_params.index.values, im_keys].values
cur_r_n1_res_df = sr.ml.gnn_gm.get_residuals(cur_r_n1_emp_gm_params, im_keys, site_col="site_id")
cur_r_n1_bias, cur_r_n1_std = cur_r_n1_res_df[im_keys].mean(axis=0), cur_r_n1_res_df[im_keys].std(axis=0)

In [ ]:
# NZGMDB 4.0 Empirical GM Parameters (CS) + NZGMDB 4.0 Observed Residuals (Shared records)
n4p0_n4p0_shared_emp_gm_params = pd.read_parquet("/Users/claudy/dev/work/data/sim_ranking/emp_gm_params/nzgmdb_v4p0/emp_gm_params.parquet")
n4p0_n4p0_shared_emp_gm_params = n4p0_n4p0_shared_emp_gm_params.rename(columns=dict(zip(sr.constants.GMM_PRED_PSA_KEYS, pred_im_keys)))
n4p0_n4p0_shared_emp_gm_params = n4p0_n4p0_shared_emp_gm_params[pred_cols]

cur_shared_records = np.intersect1d(n4p0_n4p0_shared_emp_gm_params.index, r_emp_gm_params.index)
n4p0_n4p0_shared_emp_gm_params = n4p0_n4p0_shared_emp_gm_params.loc[cur_shared_records]

n4p0_n4p0_shared_emp_gm_params[im_keys] = np.log(obs_data_v4p0.record_df.loc[n4p0_n4p0_shared_emp_gm_params.index.values, im_keys].values)
n4p0_n4p0_shared_res_df = sr.ml.gnn_gm.get_residuals(n4p0_n4p0_shared_emp_gm_params, im_keys, site_col="site_id")
n4p0_n4p0_shared_bias, n4p0_n4p0_shared_std = n4p0_n4p0_shared_res_df[im_keys].mean(axis=0), n4p0_n4p0_shared_res_df[im_keys].std(axis=0)

In [ ]:
# NZGMDB 4.0 Empirical GM Parameters (CS) + NZGMDB 4.0 Observed Residuals (All records, RJB < 400)
n4p0_n4p0_emp_gm_params = pd.read_parquet("/Users/claudy/dev/work/data/sim_ranking/emp_gm_params/nzgmdb_v4p0/emp_gm_params.parquet")
n4p0_n4p0_emp_gm_params = n4p0_n4p0_emp_gm_params.rename(columns=dict(zip(sr.constants.GMM_PRED_PSA_KEYS, pred_im_keys)))
n4p0_n4p0_emp_gm_params = n4p0_n4p0_emp_gm_params[pred_cols]
n4p0_n4p0_emp_gm_params[im_keys] = np.log(obs_data_v4p0.record_df.loc[n4p0_n4p0_emp_gm_params.index.values, im_keys].values)

n4p0_n4p0_res_df = sr.ml.gnn_gm.get_residuals(n4p0_n4p0_emp_gm_params, im_keys, site_col="site_id")
n4p0_n4p0_bias, n4p0_n4p0_std = n4p0_n4p0_res_df[im_keys].mean(axis=0), n4p0_n4p0_res_df[im_keys].std(axis=0)

In [ ]:
# NZGMDB 4.1 Empirical GM Parameters (CS) + NZGMDB 4.1 Observed Residuals (All records, RJB < 400)
n4p1_n4p1_emp_gm_params = pd.read_parquet("/Users/claudy/dev/work/data/sim_ranking/emp_gm_params/nzgmdb_v4p1/emp_gm_params_400MaxRjb.parquet")
n4p1_n4p1_emp_gm_params = n4p1_n4p1_emp_gm_params.rename(columns=dict(zip(sr.constants.GMM_PRED_PSA_KEYS, pred_im_keys)))
n4p1_n4p1_emp_gm_params = n4p1_n4p1_emp_gm_params[pred_cols]
n4p1_n4p1_emp_gm_params[im_keys] = np.log(obs_data_v4p1.record_df.loc[n4p1_n4p1_emp_gm_params.index.values, im_keys].values)

n4p1_n4p1_res_df = sr.ml.gnn_gm.get_residuals(n4p1_n4p1_emp_gm_params, im_keys, site_col="site_id")
n4p1_n4p1_bias, n4p1_n4p1_std = n4p1_n4p1_res_df[im_keys].mean(axis=0), n4p1_n4p1_res_df[im_keys].std(axis=0)

In [ ]:
# NZGMDB 4.2 Empirical GM Parameters (CS) + NZGMDB 4.2 Observed Residuals (All records, RJB < 400)
n4p2_n4p2_emp_gm_params = pd.read_parquet("/Users/claudy/dev/work/data/sim_ranking/emp_gm_params/nzgmdb_v4p2/emp_gm_params_400MaxRjb.parquet")
n4p2_n4p2_emp_gm_params = n4p2_n4p2_emp_gm_params.rename(columns=dict(zip(sr.constants.GMM_PRED_PSA_KEYS, pred_im_keys)))
n4p2_n4p2_emp_gm_params = n4p2_n4p2_emp_gm_params[pred_cols]
n4p2_n4p2_emp_gm_params[im_keys] = np.log(obs_data.record_df.loc[n4p2_n4p2_emp_gm_params.index.values, im_keys].values)

n4p2_n4p2_res_df = sr.ml.gnn_gm.get_residuals(n4p2_n4p2_emp_gm_params, im_keys, site_col="site_id")
n4p2_n4p2_bias, n4p2_n4p2_std = n4p2_n4p2_res_df[im_keys].mean(axis=0), n4p2_n4p2_res_df[im_keys].std(axis=0)

In [ ]:
# NZGMDB 4.2 Empirical GM Parameters (CS) + NZGMDB 4.2 Observed Residuals (All records, RJB < 200)
n4p2_n4p2_200_emp_gm_params = pd.read_parquet("/Users/claudy/dev/work/data/sim_ranking/emp_gm_params/nzgmdb_v4p2/emp_gm_params_200MaxRjb.parquet")
n4p2_n4p2_200_emp_gm_params = n4p2_n4p2_200_emp_gm_params.rename(columns=dict(zip(sr.constants.GMM_PRED_PSA_KEYS, pred_im_keys)))
n4p2_n4p2_200_emp_gm_params = n4p2_n4p2_200_emp_gm_params[pred_cols]
n4p2_n4p2_200_emp_gm_params[im_keys] = np.log(obs_data.record_df.loc[n4p2_n4p2_200_emp_gm_params.index.values, im_keys].values)

n4p2_n4p2_200_res_df = sr.ml.gnn_gm.get_residuals(n4p2_n4p2_200_emp_gm_params, im_keys, site_col="site_id")
n4p2_n4p2_200_bias, n4p2_n4p2_200_std = n4p2_n4p2_200_res_df[im_keys].mean(axis=0), n4p2_n4p2_200_res_df[im_keys].std(axis=0)

In [ ]:
# NZGMDB 4.2 Empirical GM Parameters (CS) + NZGMDB 4.2 Observed Residuals (GNN Records)
n4p2_n4p2_gnn_emp_gm_params = pd.read_parquet("/Users/claudy/dev/work/data/sim_ranking/emp_gm_params/nzgmdb_v4p2/emp_gm_params_400MaxRjb.parquet")
n4p2_n4p2_gnn_emp_gm_params = n4p2_n4p2_gnn_emp_gm_params.rename(columns=dict(zip(sr.constants.GMM_PRED_PSA_KEYS, pred_im_keys)))
n4p2_n4p2_gnn_emp_gm_params = n4p2_n4p2_gnn_emp_gm_params[pred_cols]
n4p2_n4p2_gnn_emp_gm_params = n4p2_n4p2_gnn_emp_gm_params.loc[gnn_val_results.index.values]

n4p2_n4p2_gnn_emp_gm_params[im_keys] = np.log(obs_data.record_df.loc[n4p2_n4p2_gnn_emp_gm_params.index.values, im_keys].values)
n4p2_n4p2_gnn_res_df = sr.ml.gnn_gm.get_residuals(n4p2_n4p2_gnn_emp_gm_params, im_keys, site_col="site_id")
n4p2_n4p2_gnn_bias, n4p2_n4p2_gnn_std = n4p2_n4p2_gnn_res_df[im_keys].mean(axis=0), n4p2_n4p2_gnn_res_df[im_keys].std(axis=0)

In [ ]:
# NZGMDB 4.0 Empirical GM Parameters (CS) + NZGMDB 4.0 Observed Residuals (GNN Records)
n4p0_n4p0_gnn_emp_gm_params = pd.read_parquet("/Users/claudy/dev/work/data/sim_ranking/emp_gm_params/nzgmdb_v4p0/emp_gm_params.parquet")
n4p0_n4p0_gnn_emp_gm_params = n4p0_n4p0_gnn_emp_gm_params.rename(columns=dict(zip(sr.constants.GMM_PRED_PSA_KEYS, pred_im_keys)))
n4p0_n4p0_gnn_emp_gm_params = n4p0_n4p0_gnn_emp_gm_params[pred_cols]

cur_shared_keys = np.intersect1d(n4p0_n4p0_gnn_emp_gm_params.index, gnn_val_results.index)
print("Shared keys: ", len(cur_shared_keys))
n4p0_n4p0_gnn_emp_gm_params = n4p0_n4p0_gnn_emp_gm_params.loc[cur_shared_keys]

n4p0_n4p0_gnn_emp_gm_params[im_keys] = np.log(obs_data_v4p0.record_df.loc[n4p0_n4p0_gnn_emp_gm_params.index.values, im_keys].values)
n4p0_n4p0_gnn_res_df = sr.ml.gnn_gm.get_residuals(n4p0_n4p0_gnn_emp_gm_params, im_keys, site_col="site_id")
n4p0_n4p0_gnn_bias, n4p0_n4p0_gnn_std = n4p0_n4p0_gnn_res_df[im_keys].mean(axis=0), n4p0_n4p0_gnn_res_df[im_keys].std(axis=0)

In [ ]:
fig, ax1, ax2, ax3, ax4 = sr.plot_utils.get_bias_residual_fig(figsize=(16, 6), std_y_axis_limits=(0, 1.5), bias_y_axis_limits=(-1.5, 1.5))

ax1.plot(periods, cur_cs_n4_bias.values, label="NZGMDB 4.2 (CS) + NZGMDB 4.2 (Shared Records)", c="lightblue")
ax1.plot(periods, cur_cs_n1_bias.values, label="NZGMDB 4.2 (CS) + NZGMDB 1.0 (Shared Records)", c="darkblue")
ax1.plot(periods, cur_r_n4_bias.values, label="Robin + NZGMDB 4.2 (Shared Records)", c="orange")
ax1.plot(periods, cur_r_n1_bias.values, label="Robin + NZGMDB 1.0 (Shared Records)", c="r")
ax1.plot(periods, n4p0_n4p0_shared_bias.values, label="NZGMDB 4.0 (CS) + NZGMDB 4.0 (Shared Records)", c="purple")
ax1.plot(periods, n4p0_n4p0_bias.values, label="NZGMDB 4.0 (CS) + NZGMDB 4.0 (All Records, $R_{JB} < 400$)", c="purple", linestyle="--")
# ax1.plot(periods, n4p1_n4p1_bias.values, label="NZGMDB 4.1 (CS) + NZGMDB 4.1 (All Records, $R_{JB} < 400$)", c="pink", linestyle="--")
ax1.plot(periods, n4p2_n4p2_bias.values, label="NZGMDB 4.2 (CS) + NZGMDB 4.2 (All Records, $R_{JB} < 400$)", c="brown", linestyle="--")
# ax1.plot(periods, n4p2_n4p2_200_bias.values, label="NZGMDB 4.2 (CS) + NZGMDB 4.2 (All Records, $R_{JB} < 200$)", c="darkgreen", linestyle="--")
ax1.plot(periods, n4p2_n4p2_gnn_bias.values, label="NZGMDB 4.2 (CS) + NZGMDB 4.2 (GNN Records)", c="blue", linestyle="dotted")
ax1.plot(periods, n4p0_n4p0_gnn_bias.values, label="NZGMDB 4.0 (CS) + NZGMDB 4.0 (GNN Records)", c="purple", linestyle="dotted")



ax1.plot(periods, mera_df.loc[im_keys, "bias"].values, label="Robin MERA", c="green")
ax1.set_xscale("log")
ax1.legend()

ax3.plot(periods, cur_cs_n4_std.values, c="lightblue")
ax3.plot(periods, cur_cs_n1_std.values, c="darkblue")
ax3.plot(periods, cur_r_n4_std.values, c="orange")
ax3.plot(periods, cur_r_n1_std.values, c="r")
ax3.plot(periods, n4p0_n4p0_shared_std.values, c="purple")
ax3.plot(periods, n4p0_n4p0_std.values, c="purple", linestyle="--")
# ax3.plot(periods, n4p1_n4p1_std.values, c="pink", linestyle="--")
ax3.plot(periods, n4p2_n4p2_std.values, c="brown",  linestyle="--")
# ax3.plot(periods, n4p2_n4p2_200_std.values, c="darkgreen", linestyle="--")
ax3.plot(periods, n4p2_n4p2_gnn_std.values, c="blue", linestyle="dotted")
ax3.plot(periods, n4p0_n4p0_gnn_std.values, c="purple", linestyle="dotted")
ax3.plot(periods, mera_df.loc[im_keys, "sigma"], c="green")
ax3.set_xscale("log")


#### Notes:
- The large difference between **CS (Shared Records)** and **Robins (Shared Records)**/**Robin MERA** appears to be due to differences in the empirical GM parameters.

### Number of Records

In [ ]:
# Number of records as a function of period
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(periods, (~cur_cs_n4_res_df[im_keys].isna()).sum(axis=0), label="CS + NZGMDB 4.2 (Shared Records)", c="lightblue")
ax.plot(periods, (~cur_cs_n1_res_df[im_keys].isna()).sum(axis=0), label="CS + NZGMDB 1.0 (Shared Records)", c="darkblue")
ax.plot(periods, (~cur_r_n4_res_df[im_keys].isna()).sum(axis=0), label="Robin + NZGMDB 4.2 (Shared Records)", c="orange")
ax.plot(periods, (~cur_r_n1_res_df[im_keys].isna()).sum(axis=0), label="Robin + NZGMDB 1.0 (Shared Records)", c="r")

ax.set_xlabel("Period (s)")
ax.set_ylabel("Number of Records")
ax.set_xscale("log")
ax.set_xlim(0.01, 10)
ax.legend()

fig.tight_layout()

### Tectonic Type

In [ ]:
n4_tect_type_count = obs_data.record_df.tect_type.value_counts()
n4_tect_type_count = n4_tect_type_count.rename(index=tect_type_mapping)

n4_shared_tect_type_count = obs_data.record_df.loc[shared_gm_records].tect_type.value_counts()
n4_shared_tect_type_count = n4_shared_tect_type_count.rename(index=tect_type_mapping)

fig, (ax1, ax2) = mlt.plotting.get_fig_axes(2, 2, 1, (8, 6))

ax1.bar(
    n4_tect_type_count.index,
    n4_tect_type_count.values,
    color="grey",
    edgecolor="black",
    linewidth=0.5,
    align="center",
)
ax1.set_title("NZGMDB 4.2 (All Records)")
ax1.set_ylabel("Count")

ax2.bar(
    n4_shared_tect_type_count.index,
    n4_shared_tect_type_count.values,
    color="grey",
    edgecolor="black",
    linewidth=0.5,
    align="center",
)
ax2.set_title("NZGMDB 4.2 (Shared Records)")
ax2.set_ylabel("Count")

fig.tight_layout()

#### NZGMDB 4.2 Empirical GM Parameters (CS) + NZGMDB 4.2 Observed Residuals - Shared Records

In [ ]:
# NZGMDB 4.2 Empirical GM Parameters (CS) + NZGMDB 4.2 Observed Residuals (Shared Records) -- Tectonic Type
cur_record_df = obs_data.record_df.loc[shared_gm_records]
cur_crustal_keys = cur_record_df.loc[cur_record_df.tect_type == "crustal"].index.values
cur_crustal_bias = cur_cs_n4_res_df.loc[cur_crustal_keys, im_keys].mean(axis=0)
cur_crustal_std = cur_cs_n4_res_df.loc[cur_crustal_keys, im_keys].std(axis=0)

cur_subduction_keys = cur_record_df.loc[cur_record_df.tect_type.isin(["subduction_interface"])].index.values
cur_subduction_bias = cur_cs_n4_res_df.loc[cur_subduction_keys, im_keys].mean(axis=0)
cur_subduction_std = cur_cs_n4_res_df.loc[cur_subduction_keys, im_keys].std(axis=0)

cur_other_keys = cur_record_df.loc[~cur_record_df.index.isin(cur_crustal_keys) & ~cur_record_df.index.isin(cur_subduction_keys)].index.values
cur_other_bias = cur_cs_n4_res_df.loc[cur_other_keys, im_keys].mean(axis=0)
cur_other_std = cur_cs_n4_res_df.loc[cur_other_keys, im_keys].std(axis=0)

In [ ]:
fig, ax1, ax2, ax3, ax4 = sr.plot_utils.get_bias_residual_fig(figsize=(16, 6), std_y_axis_limits=(0, 1.5), bias_y_axis_limits=(-1.5, 1.5))

ax1.plot(periods, cur_cs_n4_bias.values, label="All", c="black")  
ax1.plot(periods, cur_crustal_bias.values, label="Crustal", c="red")
ax1.plot(periods, cur_subduction_bias.values, label="Subduction", c="blue")
ax1.plot(periods, cur_other_bias.values, label="Other", c="orange")

ax1.plot(periods, mera_df.loc[im_keys, "bias"].values, label="Robin MERA", c="green")

ax1.set_xscale("log")
ax1.legend()

ax3.plot(periods, cur_cs_n4_std.values, c="black")
ax3.plot(periods, cur_crustal_std.values, c="red")
ax3.plot(periods, cur_subduction_std.values, c="blue")
ax3.plot(periods, cur_other_std.values, c="orange")

ax3.plot(periods, mera_df.loc[im_keys, "sigma"], c="green")
ax3.set_xscale("log")


#### NZGMDB 4.0 Empirical GM Parameters (CS) + NZGMDB 4.0 Observed Residuals - Shared Records

In [ ]:
cur_shared_records = np.intersect1d(n4p0_n4p0_shared_emp_gm_params.index, r_emp_gm_params.index)

cur_record_df = obs_data_v4p0.record_df.loc[cur_shared_records]
cur_crustal_keys = cur_record_df.loc[cur_record_df.tect_type == "crustal"].index.values
cur_crustal_bias = n4p0_n4p0_shared_res_df.loc[cur_crustal_keys, im_keys].mean(axis=0)
cur_crustal_std = n4p0_n4p0_shared_res_df.loc[cur_crustal_keys, im_keys].std(axis=0)

cur_subduction_keys = cur_record_df.loc[cur_record_df.tect_type.isin(["subduction_interface"])].index.values
cur_subduction_bias = n4p0_n4p0_shared_res_df.loc[cur_subduction_keys, im_keys].mean(axis=0)
cur_subduction_std = n4p0_n4p0_shared_res_df.loc[cur_subduction_keys, im_keys].std(axis=0)

cur_other_keys = cur_record_df.loc[~cur_record_df.index.isin(cur_crustal_keys) & ~cur_record_df.index.isin(cur_subduction_keys)].index.values
cur_other_bias = n4p0_n4p0_shared_res_df.loc[cur_other_keys, im_keys].mean(axis=0)
cur_other_std = n4p0_n4p0_shared_res_df.loc[cur_other_keys, im_keys].std(axis=0)

In [ ]:
fig, ax1, ax2, ax3, ax4 = sr.plot_utils.get_bias_residual_fig(figsize=(16, 6), std_y_axis_limits=(0, 1.5), bias_y_axis_limits=(-1.5, 1.5))

ax1.plot(periods, n4p0_n4p0_shared_bias.values, label="All", c="black")
ax1.plot(periods, cur_crustal_bias.values, label="Crustal", c="red")
ax1.plot(periods, cur_subduction_bias.values, label="Subduction", c="blue")
ax1.plot(periods, cur_other_bias.values, label="Other", c="orange")
ax1.plot(periods, mera_df.loc[im_keys, "bias"].values, label="Robin MERA", c="green")
ax1.set_xscale("log")
ax1.legend()

ax3.plot(periods, n4p0_n4p0_shared_std.values, c="black")
ax3.plot(periods, cur_crustal_std.values, c="red")
ax3.plot(periods, cur_subduction_std.values, c="blue")
ax3.plot(periods, cur_other_std.values, c="orange")
ax3.plot(periods, mera_df.loc[im_keys, "sigma"], c="green")
ax3.set_xscale("log")

## Observation Data Comparison - NZGMDB 4.2 vs NZGMDB v1.0

In [ ]:
# Plot config
plot_ims = ["pSA_0.01", "pSA_0.1", "pSA_0.5", "pSA_1.0", "pSA_3.0", "pSA_10.0"]
pred_plot_ims = [f"{cur_plot_im}_pred" for cur_plot_im in plot_ims]

In [ ]:
fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), 2, -1, ind_figsize=(8, 6))

for cur_ax, cur_im in zip(axs, plot_ims):
    cur_x_data = obs_data.record_df.loc[shared_gm_records, cur_im].values
    cur_y_data = np.exp(r_obs_df.loc[shared_gm_records, cur_im].values)
    nan_mask = np.isnan(cur_x_data) | np.isnan(cur_y_data)
    cur_x_data = cur_x_data[~nan_mask]
    cur_y_data = cur_y_data[~nan_mask]

    lims = (
        np.quantile(cur_x_data, 0.01),
        np.quantile(cur_x_data, 0.99),
    )

    cur_ax.scatter(
        cur_x_data,
        cur_y_data,
        alpha=0.5,
        s=1,
    )
    cur_ax.plot(lims, lims, color="k")

    cur_ax.set_xlabel("NZGMDB v4.2")
    cur_ax.set_ylabel("Robin (NZGMDB v1.0)")
    cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")

    cur_ax.set_xlim(lims)
    cur_ax.set_ylim(lims)
    cur_ax.set_aspect("equal")

    cur_ax.set_title(f"{cur_im} - N: {len(cur_x_data)}")

fig.tight_layout()
    


## Empirical GM Parameters Comparison

### NZGMDB v4.2 (CS) - NZGMDB v1.0 (Robin) - Shared Records

In [ ]:
fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), 2, -1, ind_figsize=(8, 6))

for cur_ax, cur_im in zip(axs, pred_plot_ims):
    cur_x_data = np.exp(emp_gm_params.loc[shared_gm_records, cur_im].values)
    cur_y_data = np.exp(r_emp_gm_params.loc[shared_gm_records, cur_im].values)

    nan_mask = np.isnan(cur_x_data) | np.isnan(cur_y_data)
    cur_x_data = cur_x_data[~nan_mask]
    cur_y_data = cur_y_data[~nan_mask]
    
    lims = (
        np.quantile(cur_x_data, 0.01),
        np.quantile(cur_x_data, 0.99),
    )

    cur_ax.scatter(
        cur_x_data,
        cur_y_data,
        alpha=0.5,
        s=1,
    )
    cur_ax.plot(lims, lims, color="k")
    cur_ax.set_xlabel("CS")
    cur_ax.set_ylabel("Robin")
    cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    cur_ax.set_xlim(lims)
    cur_ax.set_ylim(lims)
    cur_ax.set_aspect("equal")

    cur_ax.set_title(f"{cur_im} - N: {len(cur_x_data)}")

fig.tight_layout()



### NZGMDB v1.0 Inputs + CS Workflow vs Robins

In [ ]:
# ### Code for computing the GM parameters

# from pathlib import Path

# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

# import sim_ranking as sr
# import ml_tools as mlt



# robin_data_dir = Path("/Users/claudy/dev/work/data/sim_ranking/other/robin_2024_emp_gmm_study_data/Br_10_rrup_fmin_mw_cmt_HNBN_newrrup_1p0_crustal_mw3p5_rrup300_0p12")

# site_cols_map = {
#     "sta": sr.ObservedData.SiteColEnums.SITE_ID,
#     "Vs30": sr.ObservedData.SiteColEnums.VS30,
#     "Tsite": sr.ObservedData.SiteColEnums.TSITE,
#     "T0": sr.ObservedData.SiteColEnums.TSITE,
#     "Z1.0": sr.ObservedData.SiteColEnums.Z1P0,
#     "Z2.5": sr.ObservedData.SiteColEnums.Z2P5,
#     "sta_lat": sr.ObservedData.SiteColEnums.SITE_LAT,
#     "sta_lon": sr.ObservedData.SiteColEnums.SITE_LON,
# }
# event_map = {
#     "evid": sr.ObservedData.EventColEnums.EVENT_ID,
#     "mag": sr.ObservedData.EventColEnums.MAG,
#     "rake": sr.ObservedData.EventColEnums.RAKE,
#     "strike": sr.ObservedData.EventColEnums.STRIKE,
#     "dip": sr.ObservedData.EventColEnums.DIP,
#     "tect_class": sr.ObservedData.EventColEnums.TECT_TYPE,
#     "ev_depth": sr.ObservedData.EventColEnums.DEPTH,
#     "z_tor": sr.ObservedData.EventColEnums.ZTOR,
#     "ev_lat": sr.ObservedData.EventColEnums.EVENT_LAT,
#     "ev_lon": sr.ObservedData.EventColEnums.EVENT_LON,
# }
# event_site_map = {
#     "r_jb": sr.ObservedData.EventSiteColEnums.RJB,
#     "r_rup": sr.ObservedData.EventSiteColEnums.RRUP,
#     "r_x": sr.ObservedData.EventSiteColEnums.RX,
# }
# other_map = {
#     "fmin_X": sr.ObservedData.OtherColEnums.FMIN_H1,
#     "fmin_Y": sr.ObservedData.OtherColEnums.FMIN_H2,
#     "fmin_Z": sr.ObservedData.OtherColEnums.FMIN_V,
#     "score_X": sr.ObservedData.OtherColEnums.QUALITY_SCORE_H1,
#     "score_Y": sr.ObservedData.OtherColEnums.QUALITY_SCORE_H2,
#     "score_Z": sr.ObservedData.OtherColEnums.QUALITY_SCORE_V,
#     "fmin_mean_X": sr.ObservedData.OtherColEnums.FMIN_H1,
#     "fmin_mean_Y": sr.ObservedData.OtherColEnums.FMIN_H2,
#     "fmin_mean_Z": sr.ObservedData.OtherColEnums.FMIN_V,
#     "score_mean_X": sr.ObservedData.OtherColEnums.QUALITY_SCORE_H1,
#     "score_mean_Y": sr.ObservedData.OtherColEnums.QUALITY_SCORE_H2,
#     "score_mean_Z": sr.ObservedData.OtherColEnums.QUALITY_SCORE_V,
#     "is_ground_level": sr.ObservedData.OtherColEnums.IS_GROUND_LEVEL,
#     "chan": sr.ObservedData.OtherColEnums.CHANNEL,
#     "fmin_max": sr.ObservedData.OtherColEnums.FMIN,
# }
# mapping_dict = site_cols_map | event_map | event_site_map | other_map


# # Load Robin flatfile
# robin_nzgmdb_ffp = robin_data_dir / "ground_motion_im_table_rotd50_flat_variablerrup_cmt_HNBN_crustal_1p0_mw3p5_rrup300.csv"
# record_df = pd.read_csv(robin_nzgmdb_ffp, index_col=0)
# record_df = record_df.rename(columns=mapping_dict)

# assert np.all(record_df.tect_type == "Crustal")
# record_df["tect_type"] = "crustal"

# record_df.index = mlt.array_utils.numpy_str_join("_", record_df["event_id"].values.astype(str), record_df["site_id"].values.astype(str))

# im_cols = sr.constants.IMs
# cols = record_df.columns[record_df.columns.isin(sr.ObservedData.COLUMNS + im_cols)]
# record_df = record_df[cols]

# obs_data = sr.ObservedData(record_df, robin_nzgmdb_ffp, sr.constants.ObsDataSource.NZGMDB)

# sr.data.compute_nzgmdb_emp_gm_params(
#     robin_data_dir.parent / "nzgmdb_v1p0_emp_gm_params_cs_workflow.parquet",
#     obs_data,
#     300,
# )


In [ ]:
nzgmdb_v1p0_cs_workflow_emp_gm_params = pd.read_parquet(robin_data_dir.parent / "nzgmdb_v1p0_emp_gm_params_cs_workflow.parquet")

In [ ]:
fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), 2, -1, ind_figsize=(8, 6))

for cur_ax, cur_im in zip(axs, plot_ims):
    cur_x_data = np.exp(nzgmdb_v1p0_cs_workflow_emp_gm_params.loc[shared_gm_records, f"{cur_im}_mean"].values)
    cur_y_data = np.exp(r_emp_gm_params.loc[shared_gm_records, f"{cur_im}_pred"].values)

    nan_mask = np.isnan(cur_x_data) | np.isnan(cur_y_data)
    cur_x_data = cur_x_data[~nan_mask]
    cur_y_data = cur_y_data[~nan_mask]

    lims = (
        np.quantile(cur_x_data, 0.01),
        np.quantile(cur_x_data, 0.99),
    )

    cur_ax.scatter(
        cur_x_data,
        cur_y_data,
        alpha=0.5,
        s=1,
    )
    cur_ax.plot(lims, lims, color="k")

    cur_ax.set_xlabel("NZGMDB v1.0 + CS Workflow")
    cur_ax.set_ylabel("Robin")

    cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    cur_ax.set_xlim(lims)
    cur_ax.set_ylim(lims)
    cur_ax.set_aspect("equal")

    cur_ax.set_title(f"{cur_im} - N: {len(cur_x_data)}")
    
fig.tight_layout()

### NZGMDB 4.0 (CS) vs NZGMDB v1.0 (Robin) - Shared Records

In [ ]:
emp_gm_params_4p0 = pd.read_parquet("/Users/claudy/dev/work/data/sim_ranking/emp_gm_params/nzgmdb_v4p0/emp_gm_params.parquet")
cur_shared_gm_records = np.intersect1d(emp_gm_params_4p0.index, r_emp_gm_params.index)


In [ ]:
fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), 2, -1, ind_figsize=(8, 6))

for cur_ax, cur_im in zip(axs, plot_ims):
    cur_x_data = np.exp(emp_gm_params_4p0.loc[cur_shared_gm_records, f"{cur_im}_mean"].values)
    cur_y_data = np.exp(r_emp_gm_params.loc[cur_shared_gm_records, f"{cur_im}_pred"].values)

    nan_mask = np.isnan(cur_x_data) | np.isnan(cur_y_data)
    cur_x_data = cur_x_data[~nan_mask]
    cur_y_data = cur_y_data[~nan_mask]

    lims = (
        np.quantile(cur_x_data, 0.01),
        np.quantile(cur_x_data, 0.99),
    )

    cur_ax.scatter(
        cur_x_data,
        cur_y_data,
        alpha=0.5,
        s=1,
    )
    cur_ax.plot(lims, lims, color="k")
    cur_ax.set_xlabel("NZGMDB v4.0 + CS Workflow")
    cur_ax.set_ylabel("Robin")
    cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    cur_ax.set_xlim(lims)
    cur_ax.set_ylim(lims)
    cur_ax.set_aspect("equal")

    cur_ax.set_title(f"{cur_im} - N: {len(cur_x_data)}")

fig.tight_layout()




## GMM Inputs

### NZGMBD v4.2 vs NZGMDB 1.0

In [ ]:
n4_cols = ["event_lat", "event_lon", "depth", "mag", "tect_type", "dip", "rake", "ztor", "rjb", "rrup", "rx", "vs30", "z1p0", "z2p5"]
n1_cols = ["ev_lat", "ev_lon", "ev_depth", "mag", "tect_class", "dip", "rake", "z_tor", "r_jb", "r_rup", "r_x", "Vs30", "Z1.0", "Z2.5"]

plt_n4_cols = n4_cols.copy()
plt_n4_cols.remove("tect_type")

plt_n1_cols = n1_cols.copy()
plt_n1_cols.remove("tect_class")

In [ ]:
obs_data.record_df.loc[shared_gm_records, n4_cols].head(10)

In [ ]:
r_nzmgbd_df.loc[shared_gm_records, n1_cols].head(10)

In [ ]:
fig, axs = mlt.plotting.get_fig_axes(len(plt_n4_cols), 2, -1, ind_figsize=(8, 6))

for cur_n4_col, cur_n1_col, cur_ax in zip(plt_n4_cols, plt_n1_cols, axs):
    cur_x_data = obs_data.record_df.loc[shared_gm_records, cur_n4_col].values
    cur_y_data = r_nzmgbd_df.loc[shared_gm_records, cur_n1_col].values

    nan_mask = np.isnan(cur_x_data) | np.isnan(cur_y_data)
    cur_x_data = cur_x_data[~nan_mask]
    cur_y_data = cur_y_data[~nan_mask]

    lims = (
        np.quantile(cur_x_data, 0.01),
        np.quantile(cur_x_data, 0.99),
    )

    cur_ax.scatter(
        cur_x_data,
        cur_y_data,
        alpha=0.5,
        s=1,
    )
    cur_ax.plot(lims, lims, color="k")
    cur_ax.set_xlabel("NZGMDB v4.2")
    cur_ax.set_ylabel("Robin (NZGMDB v1.0)")
    cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    cur_ax.set_xlim(lims)
    cur_ax.set_ylim(lims)
    cur_ax.set_aspect("equal")

    cur_ax.set_title(f"{cur_n4_col} - N: {len(cur_x_data)}")

fig.tight_layout()

### NZGMDB 4.0 VS NZGMDB 4.2 

In [ ]:
cur_shared_records = np.intersect1d(obs_data_v4p0.record_df.index, obs_data.record_df.index)

fig, axs = mlt.plotting.get_fig_axes(len(plt_n4_cols), 2, -1, ind_figsize=(8, 6))

for cur_n4_col, cur_ax in zip(plt_n4_cols, axs):
    cur_x_data = obs_data.record_df.loc[cur_shared_records, cur_n4_col].values
    cur_y_data = obs_data_v4p0.record_df.loc[cur_shared_records, cur_n4_col].values

    nan_mask = np.isnan(cur_x_data) | np.isnan(cur_y_data)
    cur_x_data = cur_x_data[~nan_mask]
    cur_y_data = cur_y_data[~nan_mask]

    lims = (
        np.quantile(cur_x_data, 0.01),
        np.quantile(cur_x_data, 0.99),
    )

    cur_ax.scatter(
        cur_x_data,
        cur_y_data,
        alpha=0.5,
        s=1,
    )
    cur_ax.plot(lims, lims, color="k")
    cur_ax.set_xlabel("NZGMDB v4.2")
    cur_ax.set_ylabel("NZGMDB v4.0")
    cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    cur_ax.set_xlim(lims)
    cur_ax.set_ylim(lims)
    cur_ax.set_aspect("equal")

    cur_ax.set_title(f"{cur_n4_col} - N: {len(cur_x_data)}")

### NZGMDB v4.1 vs NZGMDB 4.2

In [ ]:
cur_shared_records = np.intersect1d(obs_data_v4p1.record_df.index, obs_data.record_df.index)

fig, axs = mlt.plotting.get_fig_axes(len(plt_n4_cols), 2, -1, ind_figsize=(8, 6))

for cur_n4_col, cur_ax in zip(plt_n4_cols, axs):
    cur_x_data = obs_data.record_df.loc[cur_shared_records, cur_n4_col].values
    cur_y_data = obs_data_v4p1.record_df.loc[cur_shared_records, cur_n4_col].values

    nan_mask = np.isnan(cur_x_data) | np.isnan(cur_y_data)
    cur_x_data = cur_x_data[~nan_mask]
    cur_y_data = cur_y_data[~nan_mask]


    lims = (
        np.quantile(cur_x_data, 0.01),
        np.quantile(cur_x_data, 0.99),
    )

    cur_ax.scatter(
        cur_x_data,
        cur_y_data,
        alpha=0.5,
        s=1,
    )
    cur_ax.plot(lims, lims, color="k")

    cur_ax.set_xlabel("NZGMDB v4.2")
    cur_ax.set_ylabel("NZGMDB v4.1")
    cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    cur_ax.set_xlim(lims)
    cur_ax.set_ylim(lims)
    cur_ax.set_aspect("equal")

    cur_ax.set_title(f"{cur_n4_col} - N: {len(cur_x_data)}")

fig.tight_layout()
